In [ ]:
# Load the tools needed to trace survey answers from raw data to evaluated occupation predictions.
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
from html import escape
import importlib
import subprocess
import sys
from urllib.request import urlopen

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

pd.set_option("display.html.use_mathjax", False)
pd.set_option("display.max_columns", None)


def custom_style(dataframe, *, grouped_id_column=None):
    """
    Apply the notebook’s table style and wrap grouped record IDs.
    """
    styled = dataframe.rename_axis(index=None, columns=None).style.format(precision=2, na_rep="—")
    if grouped_id_column is not None:
        styled = styled.format(escape="html", na_rep="—").set_properties(subset=[grouped_id_column], **{"max-width": "32em", "white-space": "normal"})
    elif "ID" in dataframe.columns:
        styled = styled.format({"ID": lambda value: f"{int(value)}" if pd.notna(value) else "—"})
    if isinstance(dataframe.index, pd.RangeIndex) and dataframe.index.equals(pd.RangeIndex(len(dataframe))):
        styled = styled.hide(axis="index")

    header_properties = [("background-color", "#1e293b"), ("color", "#ffffff"), ("font-family", "sans-serif"), ("font-size", "13px"), ("padding", "10px 15px"), ("text-align", "left")]
    cell_properties = [("font-family", "sans-serif"), ("font-size", "12px"), ("padding", "8px 15px"), ("border-bottom", "1px solid #e2e8f0")]
    styled = styled.set_table_styles([{"selector": "th", "props": header_properties}, {"selector": "td", "props": cell_properties}, {"selector": "tr:hover", "props": [("background-color", "#f1f5f9")]}])
    return styled.set_properties(**{"background-color": "#f8fafc"})


def style_report_text(text, role="body"):
    """
    Render report text using the style assigned to its role.
    """
    field_style = ("h3", "color: #1e293b; font-size: 18px; font-weight: 700; border-top: 1px solid #cbd5e1; padding-top: 12px; margin: 20px 0 8px")
    section_style = ("h4", "color: #0f766e; font-size: 14px; font-weight: 700; margin: 16px 0 4px")
    body_style = ("p", "color: #334155; font-size: 13px; margin: 4px 0")
    summary_style = ("p", "color: #475569; font-size: 13px; font-weight: 600; margin: 4px 0 8px")
    empty_style = ("p", "color: #64748b; font-size: 13px; font-style: italic; margin: 4px 0 8px")
    styles = {"field": field_style, "section": section_style, "body": body_style, "summary": summary_style, "empty": empty_style}
    tag, style = styles[role]
    return HTML(f"<{tag} style='font-family: sans-serif; {style}'>{escape(text)}</{tag}>")


pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

from sklearn.preprocessing import Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

In [ ]:
def transform_to_tfidf(column, feature_fit_rows, norm=None, smooth_idf=False, sublinear_tf=False, use_idf=True, field_encoders=None):
    """
    Fit TF-IDF on selected records and return one pandas column per learned token.

    Parameters
    ----------
    column : pandas.Series
        A source column with one string per record. Use empty strings for
        missing values so those records produce rows of zeros. The output
        retains the Series index and record order.

    feature_fit_rows : sequence, pandas.Index, slice, or boolean mask
        Selects the records used to learn the vocabulary and IDF weights.
        Selection uses column.loc. Integer labels refer to the Series index.
        A boolean Series must align with that index. An unindexed boolean
        mask must have one entry per record. The selected records must
        collectively contain at least one token.

    norm : {None, "l1", "l2"}, default=None
        Controls normalization after weighting. None preserves the computed
        weights. "l1" scales each nonzero row so its absolute weights sum to
        1. "l2" scales each nonzero row to unit Euclidean length. Rows of
        zeros stay zero.

    smooth_idf : bool, default=False
        Controls inverse document frequency smoothing. False uses
        log(n / df) + 1. True uses log((n + 1) / (df + 1)) + 1.
        Here n is the number of fitting records and df is the number of
        those records containing the token. This option has no effect when
        use_idf is False.

    sublinear_tf : bool, default=False
        Controls how repeated tokens count within a record. False uses the
        occurrence count. True replaces each positive count with
        1 + log(count), reducing the weight of repeated occurrences.

    use_idf : bool, default=True
        Controls whether inverse document frequency weights multiply token
        frequencies. When False, sublinear_tf and norm still apply.

    field_encoders : dict or None, default=None
        Store the fitted vectorizer by column name when collecting encoders
        for export.

    Returns
    -------
    pandas.DataFrame
        Sparse numeric features with one row per input record and one
        column per learned token. For a named Series, each feature name is
        the source column name, two underscores, and the token, such as
        ShemAvoda__nvidia. An unnamed Series uses the tokens as names.

    Notes
    -----
    The vectorizer lowercases input and includes single-character word
    tokens. Tokens absent from the fitting vocabulary contribute no features.
    """
    column = column.astype("string").fillna("").str.lower().map(lambda value: re.sub(r"[^a-zא-ת ]", "", value))

    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", norm=norm, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf, use_idf=use_idf)

    # Learn the vocabulary and weights from the fitting records only.
    vectorizer.fit(column.loc[feature_fit_rows])
    tfidf_matrix = vectorizer.transform(column)

    # Keep the source field identifiable when combining feature columns.
    column_prefix = f"{column.name}__" if column.name is not None else ""
    feature_columns = [f"{column_prefix}{token}" for token in vectorizer.get_feature_names_out()]

    features = pd.DataFrame.sparse.from_spmatrix(tfidf_matrix, index=column.index, columns=feature_columns)

    if field_encoders is not None:
        field_encoders[column.name] = vectorizer

    return features


In [ ]:
from importlib.util import find_spec

if find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2"])

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/LaBSE", tokenizer_kwargs={"do_lower_case": True}, similarity_fn_name="dot")
model.max_seq_length = 128
model.eval()

sentences = ["This is an example sentence", "Each sentence is converted"]
embeddings = model.encode(sentences, convert_to_numpy=True, show_progress_bar=False)

print("LaBSE loaded.")
print(model)
print(embeddings)


def transform_column_with_labse(column, alpha=0.5, feature_fit_rows=None):
    """
    Replace values containing Latin letters with their closest Hebrew match when the similarity exceeds alpha.
    """
    values = column.astype("string").str.strip().replace("", pd.NA)
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)
    latin_cells = values.loc[has_latin]

    # Restrict Hebrew reference values to the fitting records when supplied.
    hebrew_values = values.where(has_hebrew & ~has_latin)
    if feature_fit_rows is not None:
        hebrew_values = hebrew_values.loc[feature_fit_rows]
    hebrew_names = sorted(hebrew_values.dropna().unique())

    transformed_column = column.copy()
    if latin_cells.empty or not hebrew_names:
        return transformed_column

    # Dot products between unit embeddings give cosine similarities.
    vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    latin_count = len(latin_cells)
    similarities = vectors[:latin_count] @ vectors[latin_count:].T

    # Preserve the original value when its best match falls at or below the threshold.
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities[np.arange(latin_count), best_indices]
    best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=latin_cells.index, dtype="string")
    transformed_column.loc[latin_cells.index] = best_hebrew.where(best_scores > alpha, column.loc[latin_cells.index])

    return transformed_column# Cache the workbook bytes so rerunning the cell can reuse the download.


In [ ]:
# Cache the workbook bytes so rerunning the cell can reuse the download.
if "raw" not in globals() or raw is None:
    URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
    raw = urllib.request.urlopen(URL, timeout=90).read()

data = pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all").iloc[:200]
metadata = pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any")

display(metadata.tail(5))
display(data.tail(2))

# Retain the source rows for the remaining checks, grouping, and prediction review.
survey = data.dropna(subset=["ID"]).copy()
occupation_codes = survey["SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
feature_fit_rows = pd.Index([row_index for row_index, code in occupation_codes.dropna().items() if re.fullmatch(r"[0-9]{1,4}", code)])

text_fields = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
labse_fields = ["ShemAvoda", "SugMachlaka", "EzoAvoda", "TeurPeula"]
reference_columns = ["YeshuvAvoda"] + text_fields
field_encoders = {}

print(f"Survey records: {len(survey):,} | Metadata fields: {len(metadata):,} | Feature-fitting records: {len(feature_fit_rows):,}")

def resolve_left_references(data, value, columns):
    """
    Within each row, replace each marker with the nearest populated nonmarker value to its left.
    """
    selected = data.loc[:, columns]
    markers = selected.eq(value).fillna(False)
    blanks = selected.eq("").fillna(False)

    # Scan columns in the supplied order and carry the last usable value across each row.
    nearest_left = selected.mask(markers | blanks).ffill(axis=1)
    resolved = selected.mask(markers & nearest_left.notna(), nearest_left)

    return pd.concat([data.drop(columns=columns), resolved], axis=1).reindex(columns=data.columns)


text_transformations = {"קרית גת": "קריית גת", "לא יכול לפרט": "חסוי", "4 עונות": "ארבע עונות", 'למ"ס': "הלשכה המרכזית לסטטיסטיקה"}


def manual_explicits_transforms(value):
    """
    Apply the specified text replacements after removing the prefix and surrounding whitespace.
    """
    if isinstance(value, str):
        value = value.removeprefix("א.א.").strip()
    return text_transformations.get(value, value)

# Resolve same-row references before reviewing the values or learning features.
data = data.map(manual_explicits_transforms)
data = resolve_left_references(data, 'כנ"ל', reference_columns)

def print_free_text_status(column):
    """
    Estimate whether the column is free text by comparing one-time and recurring values.
    """
    counts = column.value_counts()
    display(style_report_text(f"{column.name} is {'a' if counts.eq(1).sum() > counts.gt(1).sum() else 'not a'} free-text feature by this heuristic."))


def print_unique_settlements(column):
    """
    Display the number of distinct nonmissing values.
    """
    display(style_report_text(f"There are {column.nunique()} unique values in {column.name}."))


def print_mandatory_status(column):
    """
    Display whether the field appears mandatory based on observed missing values.
    """
    display(style_report_text(f"{column.name} appears {'optional' if column.isna().any() else 'mandatory'} based on the recorded values."))


def print_workplace_name_counts(column):
    """
    Display counts of missing entries, nonmissing entries, and distinct values.
    """
    display(style_report_text(f"Values are missing from {column.isna().sum():,} records and present in {column.count():,} records, with {column.nunique():,} distinct values overall."))


def group_values_by_script(column):
    """
    Group nonmissing values by the presence of Hebrew and ASCII Latin letters.
    """
    values = column.astype("string")
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)

    return {"HEBREW ONLY": column.loc[has_hebrew & ~has_latin], "LATIN ONLY": column.loc[has_latin & ~has_hebrew], "MIXED HEBREW + LATIN": column.loc[has_hebrew & has_latin], "NO HEBREW OR LATIN LETTERS": column.loc[column.notna() & ~has_hebrew & ~has_latin]}


def print_script_frequencies(column, key):
    """
    Display value frequencies by script and list the record IDs matched by row index.
    """
    formatted_keys = key.astype("Int64").astype("string").fillna("—")

    for heading, values in group_values_by_script(column).items():
        record_count = len(values)
        value_count = values.nunique()
        section_summary = f"This section contains {record_count:,} {'record' if record_count == 1 else 'records'} and {value_count:,} distinct {'value' if value_count == 1 else 'values'}."
        display(style_report_text(heading, role="section"))
        display(style_report_text(section_summary, role="summary"))

        if values.empty:
            display(style_report_text("No records found.", role="empty"))
            continue

        frequencies = values.value_counts().rename_axis("Value").to_frame("Count")
        frequencies[key.name] = formatted_keys.loc[values.index].groupby(values, sort=False, observed=True).agg(", ".join)
        display(custom_style(frequencies.reset_index(), grouped_id_column=key.name))

for column_name in metadata.loc[metadata["פורמט"].eq("TEXT") & ~metadata["הסבר"].str.contains("סמל", regex=False, na=False), "משתנה"]:
    column = data[column_name]
    display(style_report_text(column_name, role="field"))
    print_free_text_status(column)
    print_unique_settlements(column)
    print_mandatory_status(column)
    print_workplace_name_counts(column)
    print_script_frequencies(column, data["ID"])


In [ ]:
for column_name in labse_fields:
    data = pd.concat([data.drop(columns=column_name), transform_column_with_labse(data[column_name], feature_fit_rows=feature_fit_rows)], axis=1).reindex(columns=data.columns)

# Record workplace replacements before the source field becomes sparse token columns.
review = survey[["ID", "ShemAvoda"]].rename(columns={"ShemAvoda": "Original workplace"})
review["Prepared workplace"] = data.loc[survey.index, "ShemAvoda"]
review = review.loc[review["Original workplace"].fillna("").ne(review["Prepared workplace"].fillna(""))]

for column_name in text_fields:
    data = pd.concat([data.drop(columns=column_name), transform_to_tfidf(data[column_name], feature_fit_rows, field_encoders=field_encoders)], axis=1)

print("Final data shape:", data.shape)

from sklearn.preprocessing import OneHotEncoder

# Encode workplace locality separately because its values identify settlements.
prepared_data = data.loc[survey.index].copy()
locality_values = prepared_data[["YeshuvAvoda"]].astype("string").fillna("")
locality_encoder = OneHotEncoder(handle_unknown="ignore")
locality_encoder.fit(locality_values.loc[feature_fit_rows])
locality_matrix = locality_encoder.transform(locality_values)
locality_features = pd.DataFrame.sparse.from_spmatrix(locality_matrix, index=survey.index, columns=locality_encoder.get_feature_names_out(["YeshuvAvoda"]))
prepared_data = pd.concat([prepared_data.drop(columns="YeshuvAvoda"), locality_features], axis=1)
field_encoders["YeshuvAvoda"] = locality_encoder

print("Prepared table after text and locality encoding:", prepared_data.shape)


In [ ]:
# What do the management codes mean and how are they distributed? Check the metadata and responses before using responsibility as context.
management_level = survey["MenahelEtMi"]
display(management_level.value_counts(dropna=False).sort_index().rename_axis("Management response").reset_index(name="Records"))

# How can management responsibility contribute without treating response codes as measured quantities? Encode the categories as categories.
prepared_data["MenahelEtMi"] = management_level.fillna(0).astype("int64")
print(f"Unanswered management responses assigned 0: {management_level.isna().sum():,}")

# Which employment statuses determine whether follow-up questions apply? Inspect their distribution before interpreting salary-source answers.
employment_status = survey["MaamadAvoda"]
display(employment_status.value_counts(dropna=False).sort_index().rename_axis("Employment status").reset_index(name="Records"))

# Can employment arrangement add context without implying numerical order? Encode the status categories separately.
employment_categories = employment_status.fillna(0).astype("Int64").astype("string")

employment_encoder = OneHotEncoder(categories=[["0", "1", "2", "3", "4", "5", "6", "7"]], handle_unknown="ignore", sparse_output=True)
employment_encoder.fit(employment_categories.loc[feature_fit_rows].to_frame("MaamadAvoda"))

employment_matrix = employment_encoder.transform(employment_categories.to_frame("MaamadAvoda"))
employment_features = pd.DataFrame.sparse.from_spmatrix(employment_matrix, index=survey.index, columns=employment_encoder.get_feature_names_out(["MaamadAvoda"]))
prepared_data = prepared_data.drop(columns="MaamadAvoda").join(employment_features)
field_encoders["MaamadAvoda"] = employment_encoder
print("MaamadAvoda one-hot matrix:", employment_matrix.shape)
print(f"Unanswered employment statuses assigned 0: {employment_status.isna().sum():,}")

# Are salary-source blanks inapplicable, unanswered, or inconsistent with employment status? Check the routing logic before preparation.
salary_source = survey["MakorSachar"]
display(salary_source.value_counts(dropna=False).sort_index().rename_axis("Salary source").reset_index(name="Records"))

salary_status_known = survey["MaamadAvoda"].notna()
salary_eligible = survey["MaamadAvoda"].isin([1, 5])
salary_ineligible = salary_status_known & ~salary_eligible
display(pd.Series({"Eligible with an answer": int((salary_eligible & salary_source.notna()).sum()), "Ineligible without an answer": int((salary_ineligible & salary_source.isna()).sum()), "Ineligible with an answer": int((salary_ineligible & salary_source.notna()).sum()), "Unknown eligibility without an answer": int((~salary_status_known & salary_source.isna()).sum())}).rename_axis("Salary eligibility check").reset_index(name="Records"))

salary_eligibility_issues = survey.loc[salary_ineligible & salary_source.notna(), ["ID", "MaamadAvoda", "MakorSachar"]]
display(salary_eligibility_issues)

# Can salary source add occupational context without treating category IDs as quantities? Encode it categorically and retain the limits of the missing-value rule.
salary_categories = salary_source.fillna(4).astype("Int64").astype("string")

salary_encoder = OneHotEncoder(categories=[["1", "2", "3", "4"]], handle_unknown="ignore", sparse_output=True)
salary_encoder.fit(salary_categories.loc[feature_fit_rows].to_frame("MakorSachar"))

salary_matrix = salary_encoder.transform(salary_categories.to_frame("MakorSachar"))
salary_features = pd.DataFrame.sparse.from_spmatrix(salary_matrix, index=survey.index, columns=salary_encoder.get_feature_names_out(["MakorSachar"]))
prepared_data = prepared_data.drop(columns="MakorSachar").join(salary_features)
field_encoders["MakorSachar"] = salary_encoder
print("MakorSachar one-hot matrix:", salary_matrix.shape)
print(f"Unanswered salary sources assigned 4: {salary_source.isna().sum():,}")

# Which qualifications were reported and which responses are absent? Inspect the field before deciding how to represent missingness.
qualifications = survey["TeudaGvoha"]
display(qualifications.value_counts(dropna=False).sort_index().rename_axis("Qualification").reset_index(name="Records"))

# Can qualifications help distinguish occupations without assuming an unanswered field means no qualification? Encode categories while identifying the substitution assumption.
qualification_categories = qualifications.fillna(9).astype("Int64").astype("string")

qualification_encoder = OneHotEncoder(categories=[["1", "2", "3", "4", "5", "6", "7", "8", "9"]], handle_unknown="ignore", sparse_output=True)
qualification_encoder.fit(qualification_categories.loc[feature_fit_rows].to_frame("TeudaGvoha"))

qualification_matrix = qualification_encoder.transform(qualification_categories.to_frame("TeudaGvoha"))
qualification_features = pd.DataFrame.sparse.from_spmatrix(qualification_matrix, index=survey.index, columns=qualification_encoder.get_feature_names_out(["TeudaGvoha"]))
prepared_data = prepared_data.drop(columns="TeudaGvoha").join(qualification_features)
field_encoders["TeudaGvoha"] = qualification_encoder
print("TeudaGvoha one-hot matrix:", qualification_matrix.shape)
print(f"Unanswered qualifications assigned 9: {qualifications.isna().sum():,}")


In [ ]:
# Are schooling years plausible, missing, or unusually low? Inspect the observed distribution before applying numerical bounds.
school_years = survey["shnotlimud"]
display(school_years.value_counts(dropna=False).sort_index().rename_axis("Schooling years").reset_index(name="Records"))

# How can models receive a complete schooling feature without learning an imputation value from evaluation records? Fit the median on the designated fitting rows.
schooling_values = school_years.where(school_years.between(8, 80)).to_frame("shnotlimud")

from sklearn.impute import SimpleImputer
schooling_imputer = SimpleImputer(strategy="median")
schooling_imputer.fit(schooling_values.loc[feature_fit_rows])

prepared_data["shnotlimud"] = schooling_imputer.transform(schooling_values).ravel()
field_encoders["shnotlimud"] = schooling_imputer
print("Schooling values replaced:", int(schooling_values["shnotlimud"].isna().sum()))
print("Training median:", schooling_imputer.statistics_[0])

# Does age provide useful context, and which values are missing or outside the selected range? Inspect before numerical preparation.
age = survey["Gil"]
display(age.value_counts(dropna=False).sort_index().rename_axis("Age").reset_index(name="Records"))

# How can missing ages be represented consistently without fitting on evaluation data? Use the designated fitting population for imputation.
age_values = age.where(age.between(18, 120)).to_frame("Gil")

age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(age_values.loc[feature_fit_rows])

prepared_data["Gil"] = age_imputer.transform(age_values).ravel()
field_encoders["Gil"] = age_imputer
print("Ages replaced:", int(age_values["Gil"].isna().sum()))
print("Training median:", age_imputer.statistics_[0])

# What does the role-start date tell us about experience, and what reference date is available? Compute elapsed time while distinguishing it from survey-time tenure.
start_dates = pd.to_datetime(survey["TarichHathala"])

reference_date = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
years_on_job = (reference_date - start_dates).dt.days / 365.25

print("Tenure reference date:", reference_date.date())
display(years_on_job.agg(["count", "min", "max", "median"]).rename_axis("Statistic").reset_index(name="Years on the job").round(2))

# How can duration values remain comparable within this run? Record the single reference date used for every conversion.
prepared_data["TarichHathala"] = years_on_job

tenure_reference_date = reference_date.date().isoformat()


In [ ]:
# Are industry codes categorical identifiers and are any incomplete? Inspect their format and coverage before encoding.
industry_codes = survey["SemelAnafSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

from openpyxl import load_workbook
source_workbook = load_workbook(io.BytesIO(raw), read_only=True, data_only=True)
source_sheet = source_workbook["Survey Dataset"]
industry_position = list(next(source_sheet.values)).index("SemelAnafSofi") + 1
short_industry_codes = industry_codes.str.fullmatch(r"[0-9]{1,3}", na=False)
industry_storage = survey.loc[short_industry_codes, ["ID", "SemelAnafSofi"]].copy()
industry_storage["Excel format"] = [source_sheet.cell(int(index) + 2, industry_position).number_format for index in industry_storage.index]
industry_storage["ID"] = industry_storage["ID"].astype("Int64")
display(industry_storage.rename(columns={"SemelAnafSofi": "Industry code"}).reset_index(drop=True))
source_workbook.close()

display(industry_codes.loc[industry_codes.str.contains("X", na=False)].value_counts().rename_axis("Partial industry code").reset_index(name="Records"))

# Does industry classification add independent context, and will it be available for new cases? Prepare it separately pending that decision.
industry_values = industry_codes.where(~industry_codes.str.fullmatch(r"[0-9]{1,4}", na=False), industry_codes.str.zfill(4)).to_frame("SemelAnafSofi")

industry_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
industry_encoder.fit(industry_values.loc[feature_fit_rows])
industry_matrix = industry_encoder.transform(industry_values)
industry_features = pd.DataFrame.sparse.from_spmatrix(industry_matrix, index=survey.index, columns=industry_encoder.get_feature_names_out(["SemelAnafSofi"]))

prepared_data = prepared_data.drop(columns="SemelAnafSofi")
field_encoders["SemelAnafSofi"] = industry_encoder
print("SemelAnafSofi one-hot matrix:", industry_matrix.shape)

# How many occupation codes are fully known, and are classes frequent enough to evaluate? Count complete labels and rare classes first.
occupation_codes = survey["SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
occupation_codes = occupation_codes.where(~occupation_codes.str.fullmatch(r"[0-9]{1,4}", na=False), occupation_codes.str.zfill(4))

complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)
class_counts = occupation_codes.loc[complete_target].value_counts()
display(class_counts.rename_axis("Occupation code").reset_index(name="Records"))
display(occupation_codes.loc[~complete_target].value_counts().rename_axis("Partial occupation code").reset_index(name="Records"))
display(pd.Series({"Complete labels": int(complete_target.sum()), "Incomplete labels": int((~complete_target).sum()), "Complete classes": class_counts.size, "Singleton classes": int(class_counts.eq(1).sum())}).rename_axis("Label summary").reset_index(name="Count"))

# Which occupation digits are known versus unresolved? Use complete codes for supervised training and preserve partial codes for consistency checks.
prepared_data["SemelMishlachSofi"] = occupation_codes
training_ids = survey.loc[complete_target, "ID"].astype("int64")
prediction_ids = survey.loc[~complete_target, "ID"].astype("int64")
y_train = occupation_codes.loc[complete_target].copy()
partial_targets = occupation_codes.loc[~complete_target].copy()

known_prefixes = partial_targets.str.split("X").str[0]
display(pd.DataFrame({"ID": prediction_ids, "Recorded target": partial_targets, "Known prefix": known_prefixes}))


In [ ]:
# Repeated text profiles can inflate random-split scores; keep identical profiles together when evaluating new wording.
from sklearn.model_selection import GroupShuffleSplit

CV_FOLDS = 3
TEST_SIZE = 0.22

text_fields = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
labse_fields = ["ShemAvoda", "SugMachlaka", "EzoAvoda", "TeurPeula"]
raw_records = survey.copy()
raw_predictors = raw_records.drop(columns=["ID", "SemelAnafSofi", "SemelMishlachSofi"])
profile_text = raw_records[text_fields].astype("string").apply(lambda column: column.str.strip().fillna(""))
profile_keys = profile_text.apply(tuple, axis=1)
profile_groups = pd.Series(pd.factorize(profile_keys)[0], index=raw_records.index)

labelled_predictors = raw_predictors.loc[complete_target]
labelled_targets = occupation_codes.loc[complete_target]
labelled_groups = profile_groups.loc[complete_target]
development_positions, test_positions = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED).split(labelled_predictors, labelled_targets, labelled_groups))
development_predictors = labelled_predictors.iloc[development_positions]
development_targets = labelled_targets.iloc[development_positions]
development_groups = labelled_groups.iloc[development_positions]
test_predictors = labelled_predictors.iloc[test_positions]
test_targets = labelled_targets.iloc[test_positions]
validation_splits = list(GroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED).split(development_predictors, development_targets, development_groups))

results_directory = Path("classification_results")
results_directory.mkdir(exist_ok=True)
split_assignments = raw_records[["ID"]].copy()
split_assignments["Group"] = profile_groups
split_assignments["Population"] = "Incomplete label"
split_assignments.loc[development_predictors.index, "Population"] = "Development"
split_assignments.loc[test_predictors.index, "Population"] = "Held-out test"
split_assignments.to_csv(results_directory / "split_assignments.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame({"Records": [len(development_targets), len(test_targets), int((~complete_target).sum())], "Complete classes": [development_targets.nunique(), test_targets.nunique(), 0]}, index=["Development", "Held-out test", "Incomplete labels"]))


In [ ]:
# Prepare one export setup for the interactive and 4K charts.
import os
import shutil
from importlib.metadata import version
from importlib.util import find_spec
from packaging.version import Version

plot_dependencies = {"plotly": "6.3.0", "kaleido": "1.0.0"}
required_plot_packages = [f"{package}>={minimum}" for package, minimum in plot_dependencies.items() if find_spec(package) is None or Version(version(package)) < Version(minimum)]
if required_plot_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *required_plot_packages])

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Use an existing Chrome or Chromium executable, or install Chrome for this runtime.
chrome_commands = ["chrome-headless-shell", "google-chrome-stable", "google-chrome", "chromium", "chromium-browser"]
chrome_path = os.environ.get("BROWSER_PATH")
if not chrome_path or not Path(chrome_path).is_file():
    chrome_path = next((path for command in chrome_commands if (path := shutil.which(command))), None)
if not chrome_path:
    chrome_path = str(pio.get_chrome())
os.environ["BROWSER_PATH"] = chrome_path

pio.renderers.default = "colab"


def save_4k_chart(figure, name):
    """
    Display an interactive chart and save standalone HTML and a 3840 × 2160 PNG.
    """
    plot_directory.mkdir(parents=True, exist_ok=True)
    download_options = {"format": "png", "filename": f"{name}_4k", "width": 1920, "height": 1080, "scale": 2}
    chart_config = {"responsive": True, "displaylogo": False, "toImageButtonOptions": download_options}
    figure.show(config=chart_config)
    figure.write_html(plot_directory / f"{name}.html", include_plotlyjs=True, config=chart_config)
    figure.write_image(plot_directory / f"{name}_4k.png", width=1920, height=1080, scale=2)


In [ ]:
# Can identical descriptions have different occupation labels? Inspect conflicts before attributing every disagreement to model failure.
analysis_values = raw_records.loc[development_predictors.index].drop(columns="ID").map(manual_explicits_transforms)
analysis_values = resolve_left_references(analysis_values, 'כנ"ל', reference_columns)
analysis_values[reference_columns] = analysis_values[reference_columns].replace("", pd.NA)
analysis_values["SemelAnafSofi"] = industry_values.loc[analysis_values.index, "SemelAnafSofi"]
analysis_values["SemelMishlachSofi"] = occupation_codes.loc[analysis_values.index]
analysis_values["TarichHathala"] = years_on_job.loc[analysis_values.index]
analysis_values["MenahelEtMi"] = analysis_values["MenahelEtMi"].fillna(0).astype("int64")
analysis_values["MaamadAvoda"] = analysis_values["MaamadAvoda"].fillna(0).astype("int64")
analysis_values["MakorSachar"] = analysis_values["MakorSachar"].fillna(4).astype("int64")
analysis_values["TeudaGvoha"] = analysis_values["TeudaGvoha"].fillna(9).astype("int64")

consistency_rows = []
value_relationships = []
for code_field in ["SemelAnafSofi", "SemelMishlachSofi"]:
    for position in range(1, 5):
        digit = analysis_values[code_field].str[position - 1]
        known_digit = digit.str.fullmatch(r"[0-9]", na=False)
        digit_values = digit.loc[known_digit].astype(int)
        digit_range = range(int(digit_values.min()), int(digit_values.max()) + 1)
        for field in analysis_values.columns.drop(code_field):
            pairs = pd.DataFrame({"Digit": digit_values, "Value": analysis_values.loc[known_digit, field].astype("string")}).dropna()
            counts = pd.crosstab(pairs["Value"], pairs["Digit"]).reindex(columns=digit_range, fill_value=0)
            totals = counts.sum(axis=1)
            distinct_digits = counts.gt(0).sum(axis=1)
            recurring = totals.ge(2)
            for value in counts.index:
                observed = counts.loc[value]
                value_relationships.append({"Code field": code_field, "Position": position, "Feature": field, "Value": value, "Occurrences": int(totals.loc[value]), "Digits": ",".join(map(str, observed.index[observed.gt(0)])), "Digit count": int(distinct_digits.loc[value]), "Recurring": bool(recurring.loc[value]), "Exclusive": bool(recurring.loc[value] and distinct_digits.loc[value] == 1)})
            for number in digit_range:
                present = counts[number].gt(0)
                repeated_values = int((present & recurring).sum())
                exclusive_values = int((present & recurring & distinct_digits.eq(1)).sum())
                percentage = 100 * exclusive_values / repeated_values if repeated_values else np.nan
                consistency_rows.append({"Code field": code_field, "Position": position, "Digit": number, "Feature": field, "Group records": int(digit_values.eq(number).sum()), "Observed records": int(counts[number].sum()), "Distinct values": int(present.sum()), "Singleton values": int((present & ~recurring).sum()), "Recurring values": repeated_values, "Exclusive recurring values": exclusive_values, "Exclusive percent": percentage})
consistency = pd.DataFrame(consistency_rows)
value_relationships = pd.DataFrame(value_relationships)
output = Path("data_preparation_results")
output.mkdir(exist_ok=True)
consistency.to_csv(output / "digit_consistency.csv", index=False, encoding="utf-8-sig")
value_relationships.to_csv(output / "digit_value_relationships.csv", index=False, encoding="utf-8-sig")

plot_directory = Path("plots")
plot_directory.mkdir(exist_ok=True)

supported_consistency = consistency.loc[consistency["Recurring values"].ge(3)].sort_values(["Exclusive percent", "Recurring values"], ascending=[False, False])
display(supported_consistency.groupby(["Code field", "Position"], sort=False).head(5))
crossing_examples = value_relationships.loc[value_relationships["Recurring"] & value_relationships["Digit count"].gt(1)].sort_values(["Digit count", "Occurrences"], ascending=[False, False])
display(crossing_examples.groupby(["Code field", "Position"], sort=False).head(5))

# Compare recurring questionnaire values within each recorded code digit.
heatmap_colors = [[0.0, "#eef4ea"], [0.45, "#b9d3ba"], [0.72, "#80b7a6"], [1.0, "#56aaa3"]]
support_columns = ["Group records", "Observed records", "Recurring values", "Exclusive recurring values"]
support_hover = "Feature: %{x}<br>Digit: %{y}<br>Records in digit: %{customdata[0]:.0f}<br>Records with a value: %{customdata[1]:.0f}<br>Recurring values: %{customdata[2]:.0f}<br>Exclusive recurring values: %{customdata[3]:.0f}"

for code_field in ["SemelAnafSofi", "SemelMishlachSofi"]:
    figure = make_subplots(rows=4, cols=1, subplot_titles=[f"Digit position {position}" for position in range(1, 5)], shared_xaxes=True, vertical_spacing=0.07)

    for position in range(1, 5):
        rows = consistency.loc[consistency["Code field"].eq(code_field) & consistency["Position"].eq(position)]
        matrix = rows.pivot(index="Digit", columns="Feature", values="Exclusive percent").reindex(columns=analysis_values.columns.drop(code_field))
        group_sizes = rows.groupby("Digit")["Group records"].first()
        digit_labels = [f"{digit} (n={group_sizes.loc[digit]})" for digit in matrix.index]
        feature_labels = matrix.columns.tolist()
        values = matrix.to_numpy(dtype=float)
        missing = np.isnan(values)
        percent_labels = np.where(missing, "", np.char.mod("%.1f%%", values))

        # Align every support count to the digit and feature shown in the corresponding cell.
        support_index = pd.MultiIndex.from_product([matrix.index, matrix.columns], names=["Digit", "Feature"])
        support = rows.set_index(["Digit", "Feature"])[support_columns].reindex(support_index).to_numpy()
        customdata = support.reshape(len(digit_labels), len(feature_labels), len(support_columns))
        hover_heading = f"Code: {code_field}<br>Position: {position}<br>" + support_hover

        # Use a separate neutral trace so missing percentages remain distinct from zero.
        figure.add_trace(go.Heatmap(z=np.where(missing, 0, np.nan), x=feature_labels, y=digit_labels, customdata=customdata, colorscale=[[0, "#e2e7e4"], [1, "#e2e7e4"]], zmin=0, zmax=1, showscale=False, text=np.where(missing, "—", ""), texttemplate="%{text}", textfont={"size": 10, "color": "#455853"}, hovertemplate=hover_heading + "<br>Exclusive share: undefined<extra></extra>", hoverongaps=False, xgap=2, ygap=2), row=position, col=1)
        figure.add_trace(go.Heatmap(z=values, x=feature_labels, y=digit_labels, customdata=customdata, coloraxis="coloraxis", text=percent_labels, texttemplate="%{text}", textfont={"size": 10, "color": "#153932"}, hovertemplate=hover_heading + "<br>Exclusive share: %{z:.2f}%<extra></extra>", hoverongaps=False, xgap=2, ygap=2), row=position, col=1)

        figure.update_xaxes(categoryorder="array", categoryarray=feature_labels, tickangle=-35, tickfont={"size": 11}, showticklabels=position == 4, automargin=True, showgrid=False, row=position, col=1)
        figure.update_yaxes(categoryorder="array", categoryarray=digit_labels, autorange="reversed", title_text="Digit and group size", automargin=True, showgrid=False, row=position, col=1)

    figure.update_layout(title={"text": f"{code_field} · Repeated values by code digit<br><sup>Share of recurring field values confined to one digit · — denotes an undefined share</sup>", "x": 0.04}, template="plotly_white", paper_bgcolor="#f8f7f1", plot_bgcolor="#f8f7f1", font={"family": "Arial, sans-serif", "size": 12, "color": "#234737"}, margin={"l": 105, "r": 145, "t": 110, "b": 100}, height=1160, autosize=True, coloraxis={"cmin": 0, "cmax": 100, "colorscale": heatmap_colors, "colorbar": {"title": {"text": "Exclusive<br>share"}, "tickvals": [0, 25, 50, 75, 100], "ticktext": ["0%", "25%", "50%", "75%", "100%"], "len": 0.82, "thickness": 15}})
    save_4k_chart(figure, f"{code_field}_digit_consistency")


In [ ]:
# Identify feature values that consistently identify all four occupation digits, not just one position.
occupation_target = analysis_values["SemelMishlachSofi"].astype("string")
four_digit_rows = []
for field in analysis_values.columns.drop(["SemelMishlachSofi", "SemelAnafSofi"], errors="ignore"):
    pairs = pd.DataFrame({"Value": analysis_values[field].astype("string"), "Target": occupation_target}).dropna()
    pairs = pairs.loc[pairs["Value"].str.strip().ne("") & pairs["Target"].str.fullmatch(r"[0-9]{4}", na=False)]
    for value, group in pairs.groupby("Value", sort=False):
        counts = group["Target"].value_counts()
        dominant_code = counts.index[0]
        four_digit_rows.append({"Feature": field, "Value": value, "Support": len(group), "Distinct targets": len(counts), "Dominant target": dominant_code, "Consistency": counts.iloc[0] / len(group), "Exclusive 4 digits": len(counts) == 1 and len(group) >= 3})
four_digit_relationships = pd.DataFrame(four_digit_rows)
four_digit_relationships.to_csv(output / "four_digit_feature_relationships.csv", index=False, encoding="utf-8-sig")
supported_relationships = four_digit_relationships.loc[four_digit_relationships["Support"].ge(3)].copy()
feature_relationship_summary = supported_relationships.groupby("Feature").agg(Supported_values=("Value", "size"), Exclusive_values=("Exclusive 4 digits", "sum"), Supporting_records=("Support", "sum")).reset_index()
feature_relationship_summary["Exclusive share"] = feature_relationship_summary["Exclusive_values"] / feature_relationship_summary["Supported_values"]
feature_relationship_summary = feature_relationship_summary.sort_values(["Exclusive share", "Supporting_records"], ascending=False)
display(feature_relationship_summary.style.hide(axis="index").format({"Exclusive share": "{:.1%}"}))
display(supported_relationships.sort_values(["Exclusive 4 digits", "Support"], ascending=False).head(30).style.hide(axis="index").format({"Consistency": "{:.1%}"}))

# Compare the share of supported values linked to one observed four-digit occupation code.
plot_rows = feature_relationship_summary.sort_values("Exclusive share", kind="stable").copy()
plot_rows["Exclusive / supported"] = plot_rows["Exclusive_values"].astype(str) + "/" + plot_rows["Supported_values"].astype(str)

figure = px.bar(plot_rows, x="Exclusive share", y="Feature", orientation="h", color="Exclusive share", color_continuous_scale="Teal", range_color=[0, 1], text="Exclusive / supported", hover_data={"Exclusive share": ":.1%", "Exclusive_values": True, "Supported_values": True, "Supporting_records": True, "Exclusive / supported": False}, labels={"Exclusive_values": "Exclusive values", "Supported_values": "Supported values", "Supporting_records": "Supporting records"})
figure.update_xaxes(range=[0, 1.16], tickvals=[0, 0.25, 0.5, 0.75, 1], tickformat=".0%", title="Share of supported values associated with one four-digit code")
figure.update_yaxes(title=None, categoryorder="array", categoryarray=plot_rows["Feature"].tolist(), automargin=True)
figure.update_traces(textposition="outside", cliponaxis=False)
figure.update_layout(title={"text": "One feature value → one four-digit occupation code<br><sup>Labels show exclusive values / supported values · support requires at least 3 records</sup>", "x": 0.04}, template="plotly_white", height=690, margin={"l": 65, "r": 100, "t": 105, "b": 65}, font={"family": "Arial, sans-serif", "size": 15, "color": "#1e293b"}, coloraxis_showscale=False)

save_4k_chart(figure, "four_digit_feature_exclusivity")


In [ ]:
# Are all feature blocks and target labels aligned to the same records? Assemble them with consistent row indices before training.
from scipy import sparse
predictor_fields = prepared_data.columns.drop(["ID", "SemelMishlachSofi"]).tolist()
X_train = prepared_data.loc[complete_target, predictor_fields].copy()
X_prediction = prepared_data.loc[~complete_target, predictor_fields].copy()

training_matrix = sparse.csr_matrix(X_train.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
prediction_matrix = sparse.csr_matrix(X_prediction.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
feature_names = np.asarray(predictor_fields)
industry_training_matrix = industry_matrix[complete_target.to_numpy()]
industry_prediction_matrix = industry_matrix[(~complete_target).to_numpy()]

display(pd.DataFrame({"Records": [len(X_train), len(X_prediction)], "Numerical features": [training_matrix.shape[1], prediction_matrix.shape[1]]}, index=["Complete target", "Incomplete target"]))

# Can the analysis be reproduced and its assumptions inspected? Save the prepared data and feature decisions.
import joblib
import sklearn
prepared_data.to_csv(output / "prepared_survey.csv", index=False, encoding="utf-8-sig")
pd.concat([training_ids.rename("ID"), X_train, y_train.rename("SemelMishlachSofi")], axis=1).to_csv(output / "training_inputs.csv", index=False, encoding="utf-8-sig")
pd.concat([prediction_ids.rename("ID"), X_prediction, partial_targets.rename("Recorded_target"), known_prefixes.rename("Known_prefix")], axis=1).to_csv(output / "prediction_inputs.csv", index=False, encoding="utf-8-sig")
sparse.save_npz(output / "training_features.npz", training_matrix)
sparse.save_npz(output / "prediction_features.npz", prediction_matrix)
sparse.save_npz(output / "industry_training_features.npz", industry_training_matrix)
sparse.save_npz(output / "industry_prediction_features.npz", industry_prediction_matrix)

pd.Series(feature_names, name="Feature").to_csv(output / "feature_names.csv", index=False, encoding="utf-8-sig")
pd.Series(industry_encoder.get_feature_names_out(["SemelAnafSofi"]), name="Feature").to_csv(output / "industry_feature_names.csv", index=False, encoding="utf-8-sig")
salary_eligibility_issues.to_csv(output / "preparation_audit.csv", index=False, encoding="utf-8-sig")
review.to_csv(output / "workplace_transformations.csv", index=False, encoding="utf-8-sig")
joblib.dump(field_encoders, output / "feature_encoders.joblib")

manifest = {"records": len(survey), "training_records": len(X_train), "prediction_records": len(X_prediction), "features": len(feature_names), "training_ids": training_ids.tolist(), "prediction_ids": prediction_ids.tolist(), "tfidf_normalization": field_encoders["ShemAvoda"].norm, "tfidf_smooth_idf": False, "tenure_reference_date": tenure_reference_date, "tenure_definition": "Elapsed days from role start to the current processing date divided by 365.25.", "tenure_assumption": "Immediate processing. The current date substitutes for the unavailable survey or last-update date.", "management_missing_value": 0, "employment_missing_category": 0, "salary_missing_category": 4, "qualification_missing_category": 9, "qualification_missing_assumption": "The participant left the self-completed question blank because there was no certificate to report.", "schooling_range": [8, 80], "age_range": [18, 120], "added_missingness_indicators": False, "industry_in_primary_model": False, "classifier_trained": False, "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__}
(output / "preparation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Prepared numerical inputs and field encoders saved.")


In [ ]:
# Reuse the numerical features already prepared, without rebuilding questionnaire columns.
import time

for package in ["xgboost", "lightgbm"]:
    if find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier

from sklearn.base import BaseEstimator, ClassifierMixin

def to_dense(matrix):
    return matrix.toarray() if sparse.issparse(matrix) else matrix

# Select development and held-out rows from the already prepared features.
development_matrix = training_matrix[development_positions]
test_matrix = training_matrix[test_positions]
assert development_matrix.shape[0] == len(development_targets) and test_matrix.shape[0] == len(test_targets)
assert development_matrix.shape[1] == test_matrix.shape[1] == prediction_matrix.shape[1]


In [ ]:
print("Exploratory evaluation. Questionnaire features were prepared before the grouped split.")

# Combine complementary classifiers to test whether their predictions improve occupation classification.
def build_ensemble():
    return VotingClassifier(estimators=[("logistic", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)), ("forest", RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1)), ("xgboost", XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=1))], voting="soft", n_jobs=1)


# Compare eight classifiers with different learning approaches on the same prepared data.
model_families = {
    "Linear SVC": (LinearSVC(class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),

    "Logistic regression": (LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), {"classifier__C": [0.1, 1]}),

    "RBF SVC": (SVC(C=1, class_weight="balanced", random_state=SEED), {}),

    "Complement NB": (ComplementNB(alpha=1), {}),

    "XGBoost": (VotingClassifier(estimators=[("xgboost", XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=SEED, n_jobs=1))], voting="soft"), {}),

    "LightGBM": (LGBMClassifier(n_estimators=40, num_leaves=7, min_child_samples=2, learning_rate=0.1, verbosity=-1, random_state=SEED, n_jobs=1), {}),

    "Random forest": (RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1), {"classifier__min_samples_leaf": [1, 2]}),

    "Soft voting": (build_ensemble(), {})
}


# Identify features associated with one occupation-code digit using training records only.
class DigitExclusiveSelector(TransformerMixin, BaseEstimator):

    def __init__(self, min_support=3):
        self.min_support = min_support

    def fit(self, X, y):
        values = sparse.csc_matrix(X)
        targets = np.asarray(y, dtype=str)
        digit_codes = np.asarray([[code[position] for position in range(4)] for code in targets])

        # Count the records containing each feature.
        self.support_ = np.diff(values.indptr)
        self.exclusive_ = np.zeros(values.shape[1], dtype=bool)

        # Identify features associated with a single digit at any occupation-code position.
        for column in np.flatnonzero(self.support_ >= self.min_support):
            rows = values.indices[values.indptr[column]:values.indptr[column + 1]]
            self.exclusive_[column] = any(np.unique(digit_codes[rows, position]).size == 1 for position in range(4))

        # Keep all features when none satisfy the selection rule.
        self.mask_ = self.exclusive_.copy()

        if not self.mask_.any():
            self.mask_[:] = True

        return self

    # Apply the selected features without using validation labels.
    def transform(self, X):
        check_is_fitted(self, "mask_")
        return X[:, self.mask_]


# Compare eight representations using normalization, digit selection, and dimensionality reduction.
from itertools import product

dataset_options = pd.DataFrame([{"Dataset": f"D{i:02d}", "L2": l2, "Digit selection": select, "SVD": svd} for i, (l2, select, svd) in enumerate(product([False, True], repeat=3), start=1)])

# Display the representations included in the experiment.
display(dataset_options.style.hide(axis="index"))

# Save the representation definitions for interpreting results.
dataset_options.to_csv(results_directory / "dataset_options.csv", index=False)


# Compare each classifier across compatible dataset representations.
parameter_grids = []
candidate_descriptions = []

for dataset in dataset_options.to_dict("records"):

    for family, (classifier, parameters) in model_families.items():

        # Complement NB requires nonnegative features, which SVD cannot guarantee.
        if dataset["SVD"] and family == "Complement NB":
            continue

        # Learn feature selection and dimensionality reduction within each training fold.
        parameter_grids.append({"normalize": [Normalizer() if dataset["L2"] else "passthrough"], "select": [DigitExclusiveSelector(min_support=3) if dataset["Digit selection"] else "passthrough"], "svd": [TruncatedSVD(n_components=16, random_state=SEED) if dataset["SVD"] else "passthrough"], "classifier": [classifier], **parameters})

        # Record the dataset, classifier, and parameters for each candidate.
        candidate_descriptions.extend([{"Dataset": dataset["Dataset"], "Classifier": family, "L2": dataset["L2"], "Digit selection": dataset["Digit selection"], "SVD": dataset["SVD"], "Parameters": json.dumps(combination)} for combination in ParameterGrid(parameters)])


# Summarize the classifier settings included in the comparison.
model_options = pd.DataFrame([{"Classifier": name, "Parameters": ", ".join(key.replace("classifier__", "") + ": " + " / ".join(map(str, values)) for key, values in grid.items()) or "Fixed"} for name, (_, grid) in model_families.items()])

# Display the classifier settings.
display(model_options.style.hide(axis="index"))

# Report the total number of datasets, candidates, and validation fits.
print("Datasets:", len(dataset_options), "| Candidates:", len(candidate_descriptions), "| CV fits:", len(candidate_descriptions) * len(validation_splits))


# Verify that the validation splits match the intended cross-validation design.
assert len(validation_splits) == CV_FOLDS, "Validation splits must match CV_FOLDS."

# Evaluate preprocessing choices and classifiers within the same pipeline.
model_pipeline = Pipeline([("normalize", "passthrough"), ("select", "passthrough"), ("svd", "passthrough"), ("classifier", LogisticRegression())])

# Prioritize macro-F1 to account for uneven occupation-class representation.
scoring = {"macro_f1": "f1_macro", "accuracy": "accuracy"}

# Select the strongest configuration using grouped validation rather than held-out test labels.
search = GridSearchCV(model_pipeline, parameter_grids, cv=validation_splits, scoring=scoring, refit="macro_f1", n_jobs=3, return_train_score=False, error_score="raise", verbose=1)

# Start measuring the full search runtime.
search_start = time.perf_counter()

# Evaluate the candidates and retain warnings for diagnosing training problems.
with warnings.catch_warnings(record=True) as recorded_warnings:
    warnings.simplefilter("always")
    with joblib.parallel_backend("threading"):
        search.fit(development_matrix, development_targets)

# Calculate the total search runtime.
search_seconds = time.perf_counter() - search_start

# Report the measured runtime.
print("Search seconds:", round(search_seconds, 1))

# Preserve the fitted search for later evaluation.
joblib.dump(search, results_directory / "search_checkpoint.joblib")


# Combine candidate descriptions with their cross-validation scores.
model_results = pd.concat([pd.DataFrame(candidate_descriptions), pd.DataFrame(search.cv_results_).drop(columns=["params"] + [name for name in search.cv_results_ if name.startswith("param_")])], axis=1)

# Save every candidate's validation results.
model_results.to_csv(results_directory / "cross_validation_results.csv", index=False, encoding="utf-8-sig")

# Summarize warnings that may indicate unstable or unsuitable configurations.
warning_counts = pd.Series([str(item.message) for item in recorded_warnings], dtype="string").value_counts().rename("Count")

# Save the warning summary for further investigation.
warning_counts.to_csv(results_directory / "training_warnings.csv", encoding="utf-8-sig")


# Identify the configuration with the highest cross-validation macro-F1.
best_configuration = model_results.loc[search.best_index_]

# Retain the strongest parameter setting for each dataset-classifier combination.
family_results = model_results.sort_values("mean_test_macro_f1", ascending=False, kind="stable").drop_duplicates(["Dataset", "Classifier"])

# Compare model performance, variability across folds, and accuracy.
ranked_models = family_results[["Dataset", "Classifier", "L2", "Digit selection", "SVD", "mean_test_macro_f1", "std_test_macro_f1", "mean_test_accuracy"]].rename(columns={"Classifier": "Model", "mean_test_macro_f1": "Macro F1", "std_test_macro_f1": "Fold SD", "mean_test_accuracy": "Accuracy"})

# Number the combinations according to their validation ranking.
ranked_models.insert(0, "Rank", range(1, len(ranked_models) + 1))

# Display the complete ranking with consistent numerical precision.
display(ranked_models.style.hide(axis="index").format({"Macro F1": "{:.3f}", "Fold SD": "{:.3f}", "Accuracy": "{:.1%}"}))

# Report the selected feature representation.
print("Selected representation:", "L2 =", best_configuration["L2"], "| Digit selection =", best_configuration["Digit selection"], "| SVD =", best_configuration["SVD"])

# Report the winning classifier and its validation performance.
print("Selected model:", best_configuration["Classifier"], "| CV macro F1:", f'{best_configuration["mean_test_macro_f1"]:.3f}', "| CV accuracy:", f'{best_configuration["mean_test_accuracy"]:.1%}')


In [ ]:
# Prepare interactive Plotly charts and 4K image export in Google Colab.
import sys
import subprocess
import importlib.util

# Install missing visualization packages.
if find_spec("plotly") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "plotly"])

# Load Plotly and configure the Colab renderer.
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"

# Ensure the output directories exist.
plot_directory.mkdir(parents=True, exist_ok=True)
results_directory.mkdir(parents=True, exist_ok=True)


# Save interactive HTML and export 4K PNG when the runtime supports it.
def save_chart(figure, name):
    figure.show()
    figure.write_html(plot_directory / f"{name}.html", include_plotlyjs=True)
    if find_spec("kaleido") is None:
        return
    try:
        figure.write_image(plot_directory / f"{name}_4k.png", width=3840, height=2160)
    except (RuntimeError, ValueError) as error:
        print(f"Static export unavailable for {name}: {error}")


# Compare the strongest classifier for each dataset representation.
dataset_results = ranked_models.sort_values("Macro F1", ascending=False).drop_duplicates("Dataset")

# Display the winning classifier and validation metrics for each dataset.
display(dataset_results[["Dataset", "Model", "L2", "Digit selection", "SVD", "Macro F1", "Fold SD", "Accuracy"]].style.hide(axis="index").format({"Macro F1": "{:.3f}", "Fold SD": "{:.3f}", "Accuracy": "{:.1%}"}))

# Save the dataset comparison.
dataset_results.to_csv(results_directory / "dataset_comparison.csv", index=False, encoding="utf-8-sig")


# Compare the 18 strongest dataset-classifier combinations.
plot_results = ranked_models.nlargest(18, "Macro F1").copy()

# Identify each combination by its dataset and classifier.
plot_results["Variant"] = plot_results["Dataset"] + " | " + plot_results["Model"]

# Place the strongest combination at the top.
plot_results = plot_results.sort_values("Macro F1")

# Show macro-F1 alongside accuracy and fold variability.
figure = px.bar(plot_results, x="Macro F1", y="Variant", orientation="h", color="Macro F1", color_continuous_scale="Teal", text="Macro F1", hover_data={"Dataset": True, "Model": True, "Fold SD": ":.3f", "Accuracy": ":.1%", "Macro F1": ":.3f", "Variant": False})

# Preserve the complete macro-F1 scale.
figure.update_xaxes(range=[0, 1.05], title="Mean cross-validation macro-F1", tickformat=".1f")

# Keep dataset and classifier labels readable.
figure.update_yaxes(title=None, automargin=True)

# Display exact validation scores.
figure.update_traces(texttemplate="%{x:.3f}", textposition="outside", cliponaxis=False)

# Apply a consistent presentation layout.
figure.update_layout(title="Occupation classifier comparison", template="plotly_white", height=850, margin=dict(l=50, r=100, t=90, b=60), coloraxis_showscale=False, font=dict(size=15))

# Display and export the comparison in interactive and 4K formats.
save_chart(figure, "classifier_comparison")


# Investigate the selected classifier's performance across validation folds.
fold_overview = pd.DataFrame([{"Fold": fold, "Training records": len(fit_rows), "Validation records": len(validation_rows), "Unseen-class records": int((~development_targets.iloc[validation_rows].isin(development_targets.iloc[fit_rows])).sum()), "Macro F1": search.cv_results_[f"split{fold - 1}_test_macro_f1"][search.best_index_], "Accuracy": search.cv_results_[f"split{fold - 1}_test_accuracy"][search.best_index_]} for fold, (fit_rows, validation_rows) in enumerate(validation_splits, start=1)])

# Preserve the fold-level results.
fold_overview.to_csv(results_directory / "selected_fold_results.csv", index=False, encoding="utf-8-sig")

# Display fold performance and unseen-class counts.
display(fold_overview.style.hide(axis="index").format({"Macro F1": "{:.3f}", "Accuracy": "{:.1%}"}))

# Prepare readable fold labels.
fold_overview["Fold label"] = "Fold " + fold_overview["Fold"].astype(str)

# Compare validation scores and inspect class coverage on hover.
figure = px.bar(fold_overview, x="Fold label", y="Macro F1", color="Macro F1", color_continuous_scale="Teal", text="Macro F1", hover_data={"Training records": True, "Validation records": True, "Unseen-class records": True, "Accuracy": ":.1%", "Macro F1": ":.3f"})

# Use the same macro-F1 scale as the classifier comparison.
figure.update_yaxes(range=[0, 1.05], title="Validation macro-F1", tickformat=".1f")

# Display exact scores above each fold.
figure.update_traces(texttemplate="%{y:.3f}", textposition="outside", cliponaxis=False)

# Keep the fold comparison readable.
figure.update_layout(title="Selected classifier: validation fold performance", template="plotly_white", height=600, margin=dict(l=60, r=60, t=90, b=60), coloraxis_showscale=False, xaxis_title=None, font=dict(size=15))

# Display and export the fold comparison.
save_chart(figure, "selected_fold_comparison")


# Investigate features associated with a single occupation-code digit.
digit_diagnostic = DigitExclusiveSelector(min_support=3).fit(development_matrix, development_targets)

# Record the observed support and digit exclusivity of each feature.
exclusive_features = pd.DataFrame({"Feature": feature_names, "Support": digit_diagnostic.support_, "Digit exclusive": digit_diagnostic.exclusive_})

# Retain features satisfying the digit-exclusivity rule.
exclusive_features = exclusive_features.loc[exclusive_features["Digit exclusive"]].sort_values("Support", ascending=False)

# Display the 25 most frequently observed exclusive features.
display(exclusive_features.head(25).style.hide(axis="index"))

# Select the 20 features with the highest observed support.
feature_plot = exclusive_features.head(20).sort_values("Support").copy()

# Visualize feature support when qualifying features exist.
if not feature_plot.empty:

    # Compare feature occurrence counts without treating them as predictive importance.
    figure = px.bar(feature_plot, x="Support", y="Feature", orientation="h", color="Support", color_continuous_scale="Teal", text="Support", hover_data={"Feature": True, "Support": True})

    # Explain that support counts development records.
    figure.update_xaxes(title="Development records containing the feature", rangemode="tozero", dtick=1)

    # Preserve readable feature names.
    figure.update_yaxes(title=None, automargin=True)

    # Display exact support counts.
    figure.update_traces(textposition="outside", cliponaxis=False)

    # Apply the same visual style as the other charts.
    figure.update_layout(title="Features associated with a single occupation-code digit", template="plotly_white", height=850, margin=dict(l=50, r=100, t=90, b=60), coloraxis_showscale=False, font=dict(size=15))

    # Display and export the feature-support chart.
    save_chart(figure, "digit_exclusive_features")

else:

    # Report when no features meet the minimum support requirement.
    print("No digit-exclusive features met the minimum support requirement.")


In [ ]:
# Supply a score ranking only when the fitted classifier exposes one.
def prediction_scores(fitted_pipeline, inputs):
    if hasattr(fitted_pipeline, "predict_proba"):
        return fitted_pipeline.predict_proba(inputs), "Model probabilities (uncalibrated)"
    if hasattr(fitted_pipeline, "decision_function"):
        scores = np.asarray(fitted_pipeline.decision_function(inputs))
        return scores if scores.ndim == 2 else np.column_stack([-scores, scores]), "Decision scores (not probabilities)"
    return None, "Unavailable"

# Evaluate the selected classifier on held-out records excluded from model selection.
selected_pipeline = search.best_estimator_

# Predict complete four-digit occupation codes.
test_predictions = np.asarray(selected_pipeline.predict(test_matrix)).astype(str).ravel()

# Standardize recorded labels for comparisons.
actual_targets = np.asarray(test_targets).astype(str).ravel()

# Compare against the most frequent occupation in development data.
baseline = DummyClassifier(strategy="most_frequent").fit(development_matrix, development_targets)

# Predict the held-out records using the baseline.
baseline_predictions = np.asarray(baseline.predict(test_matrix)).astype(str).ravel()


# Retain original questionnaire responses for reviewing prediction errors.
held_out_predictions = raw_records.loc[test_predictors.index, ["ID", "ShemAvoda", "EzoAvoda"]].copy()

# Attach actual and predicted occupation codes.
held_out_predictions["Recorded target"] = actual_targets
held_out_predictions["Prediction"] = test_predictions

# Distinguish previously observed occupations from unseen classes.
held_out_predictions["Class seen in development"] = np.isin(actual_targets, np.asarray(development_targets).astype(str))

# Identify correct predictions.
held_out_predictions["Correct"] = actual_targets == test_predictions

# Separate errors on unseen classes from errors on known classes.
held_out_predictions["Error type"] = np.select([held_out_predictions["Correct"], ~held_out_predictions["Class seen in development"]], ["Correct", "Unseen occupation"], default="Incorrect known occupation")


# Compare held-out accuracy and macro-F1 with the majority baseline.
held_out_metrics = pd.DataFrame([{"Model": best_configuration["Classifier"], "Accuracy": accuracy_score(actual_targets, test_predictions), "Macro F1": f1_score(actual_targets, test_predictions, average="macro", zero_division=0)}, {"Model": "Majority baseline", "Accuracy": accuracy_score(actual_targets, baseline_predictions), "Macro F1": f1_score(actual_targets, baseline_predictions, average="macro", zero_division=0)}])

# Save and display the held-out model comparison.
held_out_metrics.to_csv(results_directory / "held_out_metrics.csv", index=False, encoding="utf-8-sig")
display(held_out_metrics.style.hide(axis="index").format({"Accuracy": "{:.1%}", "Macro F1": "{:.3f}"}))


# Measure agreement across successive occupation-code prefixes.
prefix_metrics = pd.DataFrame([{"Digits": width, "Agreement": np.mean([predicted[:width] == actual[:width] for predicted, actual in zip(test_predictions, actual_targets)])} for width in range(1, 5)])

# Save and display hierarchical agreement.
prefix_metrics.to_csv(results_directory / "hierarchical_agreement.csv", index=False, encoding="utf-8-sig")
display(prefix_metrics.style.hide(axis="index").format({"Agreement": "{:.1%}"}))


# Measure agreement at each individual occupation-code position.
digit_metrics = pd.DataFrame([{"Digit": position + 1, "Accuracy": np.mean([predicted[position] == actual[position] for predicted, actual in zip(test_predictions, actual_targets)])} for position in range(4)])

# Save and display digit-level accuracy.
digit_metrics.to_csv(results_directory / "digit_accuracy.csv", index=False, encoding="utf-8-sig")
display(digit_metrics.style.hide(axis="index").format({"Accuracy": "{:.1%}"}))


# Inspect class-level precision, recall, F1, and support.
classification_details = pd.DataFrame(classification_report(actual_targets, test_predictions, output_dict=True, zero_division=0)).T

# Save and display the classification report.
classification_details.to_csv(results_directory / "classification_report.csv", encoding="utf-8-sig")
display(classification_details.round(3))


# Report how many held-out records belong to occupations absent from development data.
unseen_count = int((~held_out_predictions["Class seen in development"]).sum())
print("Held-out records:", len(actual_targets))
print("Records with unseen occupations:", unseen_count)
print("Distinct unseen occupations:", len(set(actual_targets) - set(np.asarray(development_targets).astype(str))))


# Retrieve class scores when the selected model supports ranking alternatives.
test_scores, test_score_kind = prediction_scores(selected_pipeline, test_matrix)

# Evaluate whether the correct occupation appears among the top three alternatives.
if test_scores is not None:

    # Match score columns to their occupation labels.
    score_classes = np.asarray(selected_pipeline.classes_).astype(str)

    # Rank up to three candidate occupations for each record.
    top_three = score_classes[np.argsort(-test_scores, axis=1, kind="stable")[:, :min(3, len(score_classes))]]

    # Record the alternatives and whether they contain the correct occupation.
    held_out_predictions["Top three"] = [", ".join(values) for values in top_three]
    held_out_predictions["Top-three correct"] = [actual in alternatives for actual, alternatives in zip(actual_targets, top_three)]

    # Report top-three accuracy without assuming the scores are calibrated probabilities.
    print("Top-three accuracy:", f'{held_out_predictions["Top-three correct"].mean():.1%}')

# Identify the type of score used for ranking.
held_out_predictions["Score kind"] = test_score_kind

# Save and display held-out predictions for manual review.
held_out_predictions.to_csv(results_directory / "held_out_predictions.csv", index=False, encoding="utf-8-sig")
display(held_out_predictions)


# Examine incorrect classifications alongside the original occupation descriptions.
incorrect_records = raw_records.loc[held_out_predictions.index[~held_out_predictions["Correct"]], ["ID", "EzoAvoda", "TeurPeula", "TeurTafkid", "SemelMishlachSofi"]].join(held_out_predictions[["Prediction", "Class seen in development", "Error type"]])

# Display records requiring error analysis.
display(incorrect_records)


In [ ]:
# How can incomplete occupation codes be predicted without contradicting known digits? Refit on complete labels and check each partial-code constraint.
final_pipeline = clone(selected_pipeline).fit(training_matrix, labelled_targets)
unresolved_predictors = prediction_matrix
final_predictions = np.asarray(final_pipeline.predict(prediction_matrix)).ravel()
prediction_results = raw_records.loc[~complete_target, ["ID", "ShemAvoda", "EzoAvoda"]].copy()
prediction_results["Recorded target"] = occupation_codes.loc[~complete_target]
prediction_results["Known prefix"] = known_prefixes
prediction_results["Predicted occupation"] = final_predictions
prediction_results["Prefix agrees"] = [prediction.startswith(prefix) for prediction, prefix in zip(final_predictions, known_prefixes)]


partial_scores, partial_score_kind = prediction_scores(final_pipeline, prediction_matrix)
trained_codes = final_pipeline.classes_
ranking_rows = []
for row_number, (index, record) in enumerate(prediction_results.iterrows()):
    compatible = np.array([all(original == "X" or original == proposed for original, proposed in zip(record["Recorded target"], code)) for code in trained_codes])
    ranking = np.argsort(-partial_scores[row_number], kind="stable") if partial_scores is not None else np.array([], dtype=int)
    compatible_ranking = ranking[compatible[ranking]]
    ranking_rows.append({"ID": int(record["ID"]), "Top three": ", ".join(trained_codes[ranking[:3]]) if len(ranking) else "", "Score kind": partial_score_kind, "Maximum score": float(partial_scores[row_number].max()) if partial_scores is not None else np.nan, "Digit-compatible alternative": trained_codes[compatible_ranking[0]] if len(compatible_ranking) else pd.NA, "Compatible learned classes": int(compatible.sum())})
prediction_results = prediction_results.merge(pd.DataFrame(ranking_rows), on="ID", how="left")
prediction_results.to_csv(results_directory / "incomplete_label_predictions.csv", index=False, encoding="utf-8-sig")
display(prediction_results)

# The final classifier expects the already transformed columns, not a raw questionnaire.
print("Final classifier input features:", training_matrix.shape[1])
print("Incomplete records awaiting classification:", prediction_matrix.shape[0])
display(prediction_results[["ID", "Recorded target", "Predicted occupation", "Digit-compatible alternative"]])


In [ ]:
# What must be saved so later predictions use the same feature definitions? Persist the fitted model and preprocessing artifacts.
joblib.dump(final_pipeline, results_directory / "selected_pipeline.joblib")
pd.Series(feature_names, name="Feature").to_csv(results_directory / "selected_feature_names.csv", index=False, encoding="utf-8-sig")
run_manifest = {"seed": SEED, "training_records": len(labelled_predictors), "development_records": len(development_predictors), "test_records": len(test_predictors), "incomplete_records": len(prediction_results), "folds": len(validation_splits), "candidates": len(model_results), "selected_classifier": best_configuration["Classifier"], "selected_normalization": "l2" if best_configuration["L2"] else None, "selected_digit_selection": bool(best_configuration["Digit selection"]), "selected_svd": bool(best_configuration["SVD"]), "selected_parameters": json.loads(best_configuration["Parameters"]), "validation_macro_f1": float(best_configuration["mean_test_macro_f1"]), "test_accuracy": float(held_out_metrics.iloc[0]["Accuracy"]), "test_macro_f1": float(held_out_metrics.iloc[0]["Macro F1"]), "final_features": training_matrix.shape[1], "tenure_reference_date": tenure_reference_date, "management_missing_value": 0, "employment_missing_category": 0, "salary_missing_category": 4, "qualification_missing_category": 9, "text_transformations": text_transformations, "industry_in_primary_model": False, "sklearn_version": sklearn.__version__, "search_seconds": search_seconds}
(results_directory / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Selected pipeline, validation results, and final predictions saved.")

# Show held-out agreement by prefix alongside the selected model and majority baseline.
figure = make_subplots(rows=1, cols=2, subplot_titles=["Agreement by occupation-code prefix", "Selected model and majority baseline"], horizontal_spacing=0.13)

prefix_labels = prefix_metrics["Digits"].astype(str).tolist()
figure.add_trace(go.Bar(x=prefix_labels, y=prefix_metrics["Agreement"], marker_color="#234737", texttemplate="%{y:.1%}", textposition="outside", cliponaxis=False, showlegend=False, hovertemplate="Prefix length: %{x}<br>Agreement: %{y:.1%}<extra></extra>"), row=1, col=1)

for metric, color in [("Accuracy", "#234737"), ("Macro F1", "#946E37")]:
    figure.add_trace(go.Bar(name=metric, x=held_out_metrics["Model"], y=held_out_metrics[metric], marker_color=color, texttemplate="%{y:.3f}", textposition="outside", cliponaxis=False, hovertemplate="%{x}<br>" + metric + ": %{y:.3f}<extra></extra>"), row=1, col=2)

figure.update_xaxes(title="Occupation prefix length", type="category", categoryorder="array", categoryarray=prefix_labels, row=1, col=1)
figure.update_xaxes(title=None, automargin=True, row=1, col=2)
figure.update_yaxes(range=[0, 1.10], tickvals=[0, 0.25, 0.5, 0.75, 1], tickformat=".0%", title="Agreement", row=1, col=1)
figure.update_yaxes(range=[0, 1.10], tickvals=[0, 0.25, 0.5, 0.75, 1], tickformat=".2f", title="Score", row=1, col=2)
figure.update_layout(title={"text": "Occupation classification on the held-out records", "x": 0.04}, template="plotly_white", barmode="group", height=630, margin={"l": 65, "r": 55, "t": 105, "b": 95}, font={"family": "Arial, sans-serif", "size": 15, "color": "#1e293b"}, legend={"orientation": "h", "x": 1, "xanchor": "right", "y": -0.17, "yanchor": "top"})

save_4k_chart(figure, "held_out_summary")
